Create a csv file with the county FIPS code and the following county-level census data:
- 2022 Land area in square kilometers
- 2022 Water area in square kilometers
- 2022 Farm area in square kilometers
- 2022 Crop area in square kilometers
- 2020 Population
- 2023 RUCC scores

In [11]:
# geopandas not included in erdos environment, need to download
# Download Census TIGER/Line shapefules here: https://www.census.gov/geographies/mapping-files/time-series/geo/tiger-line-file.html

import geopandas as gpd
import pandas as pd

counties = gpd.read_file('../../work/tl_2022_us_county/tl_2022_us_county.shp')


In [12]:
counties = counties.rename(
    columns = {
        'GEOID': 'FIPS',
    }
)

counties.head()

,STATEFP,COUNTYFP,COUNTYNS,FIPS,NAME,NAMELSAD,LSAD,CLASSFP,MTFCC,CSAFP,CBSAFP,METDIVFP,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry
0,31,039,00835841,31039,Cuming,Cuming County,06,H1,G4020,None,None,None,A,1477644346,10691216,+41.9158651,-096.7885168,"POLYGON ((-96.55516 41.91587, -96.55515 41.914..."
1,53,069,01513275,53069,Wahkiakum,Wahkiakum County,06,H1,G4020,None,None,None,A,680980770,61564427,+46.2946377,-123.4244583,"POLYGON ((-123.72755 46.2645, -123.72756 46.26..."
2,35,011,00933054,35011,De Baca,De Baca County,06,H1,G4020,None,None,None,A,6016818946,29090018,+34.3592729,-104.3686961,"POLYGON ((-104.89337 34.08894, -104.89337 34.0..."
3,31,109,00835876,31109,Lancaster,Lancaster County,06,H1,G4020,None,None,None,A,2169272978,22847034,+40.7835474,-096.6886584,"POLYGON ((-96.68493 40.5233, -96.69219 40.5231..."
4,31,129,00835886,31129,Nuckolls,Nuckolls County,06,H1,G4020,None,None,None,A,1489645185,1718484,+40.1764918,-098.0468422,"POLYGON ((-98.2737 40.1184, -98.27374 40.1224,..."


In [13]:
RUCC_df = pd.read_excel('data/Ruralurbancontinuumcodes2023.xlsx', dtype={'FIPS': str})

RUCC_df.head()

,FIPS,State,County_Name,Population_2020,RUCC_2023,Description
0,01001,AL,Autauga County,58805,2.0,"Metro - Counties in metro areas of 250,000 to ..."
1,01003,AL,Baldwin County,231767,3.0,Metro - Counties in metro areas of fewer than ...
2,01005,AL,Barbour County,25223,6.0,"Nonmetro - Urban population of 5,000 to 20,000..."
3,01007,AL,Bibb County,22293,1.0,Metro - Counties in metro areas of 1 million p...
4,01009,AL,Blount County,59134,1.0,Metro - Counties in metro areas of 1 million p...


In [14]:
xls = pd.ExcelFile('data/NASSAgcensusDownload2022.xlsx')

crop_df = pd.read_excel(xls, 'Farms', dtype={'FIPSTEXT': str})

crop_df['FIPSTEXT']


0       00000
1       01001
2       01003
3       01005
4       01007
        ...  
3075    56037
3076    56039
3077    56041
3078    56043
3079    56045
Name: FIPSTEXT, Length: 3080, dtype: str

In [15]:
desired_columns = ['FIPSTEXT','y22_M050_valueNumeric', 'y22_M052_valueNumeric']

crop_df_acres = crop_df[desired_columns].rename(
    columns = {
        'FIPSTEXT': 'FIPS',
        'y22_M050_valueNumeric': 'farm_acre_as_percent',
        'y22_M052_valueNumeric': 'crop_acre_as_percent'
    }
)

crop_df_acres.head()

,FIPS,farm_acre_as_percent,crop_acre_as_percent
0,00000,38.9,16.91
1,01001,27.5,9.33
2,01003,17.8,10.24
3,01005,38.0,9.79
4,01007,10.6,2.87


In [16]:
all_data = pd.merge(counties, crop_df_acres, on='FIPS')
all_data = pd.merge(all_data, RUCC_df, on='FIPS')

all_data.head()

,STATEFP,COUNTYFP,COUNTYNS,FIPS,NAME,NAMELSAD,LSAD,CLASSFP,MTFCC,CSAFP,...,INTPTLAT,INTPTLON,geometry,farm_acre_as_percent,crop_acre_as_percent,State,County_Name,Population_2020,RUCC_2023,Description
0,31,039,00835841,31039,Cuming,Cuming County,06,H1,G4020,None,...,+41.9158651,-096.7885168,"POLYGON ((-96.55516 41.91587, -96.55515 41.914...",99.2,86.60,NE,Cuming County,9013,9.0,"Nonmetro - Urban population of fewer than 5,00..."
1,53,069,01513275,53069,Wahkiakum,Wahkiakum County,06,H1,G4020,None,...,+46.2946377,-123.4244583,"POLYGON ((-123.72755 46.2645, -123.72756 46.26...",6.3,1.83,WA,Wahkiakum County,4422,8.0,"Nonmetro - Urban population of fewer than 5,00..."
2,35,011,00933054,35011,De Baca,De Baca County,06,H1,G4020,None,...,+34.3592729,-104.3686961,"POLYGON ((-104.89337 34.08894, -104.89337 34.0...",62.5,0.99,NM,De Baca County,1698,9.0,"Nonmetro - Urban population of fewer than 5,00..."
3,31,109,00835876,31109,Lancaster,Lancaster County,06,H1,G4020,None,...,+40.7835474,-096.6886584,"POLYGON ((-96.68493 40.5233, -96.69219 40.5231...",76.0,65.71,NE,Lancaster County,322608,2.0,"Metro - Counties in metro areas of 250,000 to ..."
4,31,129,00835886,31129,Nuckolls,Nuckolls County,06,H1,G4020,None,...,+40.1764918,-098.0468422,"POLYGON ((-98.2737 40.1184, -98.27374 40.1224,...",86.9,60.00,NE,Nuckolls County,4095,9.0,"Nonmetro - Urban population of fewer than 5,00..."


In [17]:
county_area = all_data[["FIPS", "ALAND", 'AWATER', 'farm_acre_as_percent', 'crop_acre_as_percent', 'Population_2020', 'RUCC_2023']].copy()

county_area["area_land_km2"] = (
    county_area["ALAND"] / 1_000_000
)

county_area["area_water_km2"] = (
    county_area["AWATER"] / 1_000_000
)

county_area['farm_area_km2'] = (
    county_area['area_land_km2'] * county_area['farm_acre_as_percent'] / 100
)

county_area['crop_area_km2'] = (
    county_area['area_land_km2'] * county_area['crop_acre_as_percent'] / 100
)




In [21]:
county_area.sample(10)

,FIPS,ALAND,AWATER,farm_acre_as_percent,crop_acre_as_percent,Population_2020,RUCC_2023,area_land_km2,area_water_km2,farm_area_km2,crop_area_km2
1652,28041,1845983898,15430619,13.1,2.96,13530,8.0,1845.983898,15.430619,241.823891,54.641123
250,37063,744736824,26723652,8.8,3.85,324833,2.0,744.736824,26.723652,65.536841,28.672368
422,29119,1397155454,520251,55.4,11.77,23303,8.0,1397.155454,0.520251,774.024122,164.445197
290,37115,1164498325,4840167,14.9,2.74,21193,2.0,1164.498325,4.840167,173.510250,31.907254
451,34039,266163569,6841269,0.2,0.10,575345,1.0,266.163569,6.841269,0.532327,0.266164
525,37093,1010491623,3949309,14.7,8.11,52082,2.0,1010.491623,3.949309,148.542269,81.950871
2592,21215,483473767,12916810,49.0,23.47,19490,1.0,483.473767,12.916810,236.902146,113.471293
1249,48167,982664550,1280836456,14.1,3.41,350682,1.0,982.664550,1280.836456,138.555702,33.508861
902,08045,7633799008,21704895,21.3,3.37,61685,5.0,7633.799008,21.704895,1625.999189,257.259027
2450,48279,2631897589,3973764,78.7,63.55,13045,6.0,2631.897589,3.973764,2071.303403,1672.570918


In [19]:
county_area.to_csv("../data/county_data_2022.csv", index=False)